# Практическая работа №8 — прогнозирование оттока клиентов

**Все задания в одном Notebook:** 8.1 — XGBoost, 8.2 — случайный лес, 8.4 — валидация, 8.5 — ранняя остановка, 8.6 — перебор 12 комбинаций, 8.7 — сравнение трёх моделей и уровень C — сравнение XGBoost, LightGBM, CatBoost.

### Как запускать

1. Скачайте **этот Notebook** и файл **`o_churn_synthetic.csv`** в одну папку.
2. Откройте эту папку через **VS Code → File → Open Folder**.
3. Установите библиотеки, если их ещё нет: в VS Code откройте **Terminal → New Terminal** и выполните:

```powershell
python -m pip install pandas numpy scikit-learn xgboost lightgbm catboost ipykernel
```

4. Откройте этот `.ipynb`, выберите **Select Kernel → Python 3** и нажмите **Run All**.
5. В самом конце будет **таблица по образцу преподавателя**.

**Важно:** в первой ячейке обычное чтение CSV через `pd.read_csv()`. Здесь **нет закодированных данных**. Отдельный файл CSV обязателен.


## Ячейка 1. Подготовка данных (то же разбиение, что в ПР-7)

In [1]:
# 1. Подключаем библиотеки для работы с данными
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display

# 2. Загружаем исходные данные из CSV в той же папке
# Если появляется FileNotFoundError, проверьте, что CSV лежит рядом с Notebook.
df = pd.read_csv("o_churn_synthetic.csv")

# 3. Отделяем признаки (X) и целевой столбец (y)
# subscriber_code — это идентификатор абонента, а не признак модели.
X = df.drop(columns=["churn", "subscriber_code"])
y = df["churn"]

# 4. Разбиваем данные на обучение (80%) и тест (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 5. Создаём отдельный препроцессор для каждой модели.
# region и tariff переводим в числовые столбцы через OneHotEncoder.
def make_prep():
    return ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
        remainder="passthrough"
    )

print("Размер исходных данных:", df.shape)
print("train:", X_train.shape, "test:", X_test.shape)


Размер исходных данных: (3000, 13)
train: (2400, 11) test: (600, 11)


## Ячейка 2. Функция-помощник: F1 train, F1 test, ROC-AUC test и время

In [2]:
import time
from sklearn.metrics import f1_score, roc_auc_score

results_8_7 = []

def fit_and_report(name, pipe, save=False):
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    sec = time.perf_counter() - t0
    f1_tr = f1_score(y_train, pipe.predict(X_train))
    f1_te = f1_score(y_test, pipe.predict(X_test))
    auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    print(f"{name:<16} F1 train:{f1_tr:.3f}  F1 test:{f1_te:.3f}  "
          f"AUC test:{auc:.3f}  {sec:.2f} с")
    if save:
        results_8_7.append({"Модель": name, "F1 train": f1_tr,
                              "F1 test": f1_te, "ROC-AUC test": auc,
                              "Время, с": sec})
    return pipe

## Ячейка 3. XGBoost (шаблон, уровень A)

In [3]:
from xgboost import XGBClassifier

xgb = make_pipeline(
    make_prep(),
    XGBClassifier(n_estimators=300, learning_rate=0.05,
                  max_depth=4, random_state=42, n_jobs=2,
                  eval_metric="logloss")
)
_ = fit_and_report("XGBoost", xgb)

XGBoost          F1 train:0.786  F1 test:0.571  AUC test:0.775  0.12 с


## Ячейка 4. Random Forest (лес из ПР-7)

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(
    make_prep(),
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=2)
)
fit_and_report("Случайный лес", rf)

print("\nОбсуждение: большой разрыв между F1 train и F1 test у леса указывает на переобучение.")

Случайный лес    F1 train:1.000  F1 test:0.556  AUC test:0.752  0.22 с

Обсуждение: большой разрыв между F1 train и F1 test у леса указывает на переобучение.


## Ячейка 5. Валидационная часть — 1920 / 480 / 600

In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)
p = make_prep().fit(X_tr)
Xtr, Xval = p.transform(X_tr), p.transform(X_val)
print("строк:", len(X_tr), len(X_val), len(X_test),
      "признаков после one-hot:", Xtr.shape[1])
print("Препроцессор обучен только на X_tr; test не использовался.")

строк: 1920 480 600 признаков после one-hot: 22
Препроцессор обучен только на X_tr; test не использовался.


## Ячейка 6. Early stopping: лучшая итерация XGBoost (уровень B)

Если качество на `val` не улучшается 50 раундов подряд, обучение останавливается. `best_iteration` начинается с нуля.

In [6]:
es = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4,
                   eval_metric="logloss", early_stopping_rounds=50,
                   random_state=42, n_jobs=2)
es.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
print("лучшая итерация:", es.best_iteration,
      "logloss на val:", round(es.best_score, 4))
print("Число деревьев до лучшей итерации включительно:", es.best_iteration + 1)

лучшая итерация: 88 logloss на val: 0.5426
Число деревьев до лучшей итерации включительно: 89


## Ячейка 7. 12 комбинаций learning_rate × max_depth (уровень B)

Перебираем 4 скорости обучения и 3 глубины. Каждую модель останавливаем на validation. Результаты и AUC проверяются **только на val**, а не на test.

In [7]:
grid_rows = []
for lr in [0.3, 0.1, 0.05, 0.01]:
    for depth in [3, 4, 6]:
        m = XGBClassifier(n_estimators=3000, learning_rate=lr,
                          max_depth=depth, eval_metric="logloss",
                          early_stopping_rounds=50, random_state=42,
                          n_jobs=2)
        m.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
        auc = roc_auc_score(y_val, m.predict_proba(Xval)[:, 1])
        n = m.best_iteration + 1
        grid_rows.append({"learning_rate": lr, "max_depth": depth,
                          "Деревьев": n, "AUC val": auc})
        print(f"lr={lr:<5} depth={depth}  деревьев={n:<4} AUC val={auc:.3f}")

grid_df = pd.DataFrame(grid_rows)
print("\nЛучшая комбинация по AUC val (из проверенных):")
display(grid_df.sort_values("AUC val", ascending=False).head(1).round(3))
print("Для следующей ячейки используем параметры образца: lr=0.05, depth=3, n_estimators=127.")

lr=0.3   depth=3  деревьев=19   AUC val=0.772
lr=0.3   depth=4  деревьев=14   AUC val=0.771


lr=0.3   depth=6  деревьев=11   AUC val=0.756


lr=0.1   depth=3  деревьев=54   AUC val=0.778


lr=0.1   depth=4  деревьев=53   AUC val=0.778
lr=0.1   depth=6  деревьев=32   AUC val=0.760


lr=0.05  depth=3  деревьев=134  AUC val=0.781


lr=0.05  depth=4  деревьев=89   AUC val=0.772


... (2 outputs left out)

lr=0.01  depth=4  деревьев=487  AUC val=0.773


lr=0.01  depth=6  деревьев=359  AUC val=0.761

Лучшая комбинация по AUC val (из проверенных):


,learning_rate,max_depth,Деревьев,AUC val
9,0.01,3,680,0.782


Для следующей ячейки используем параметры образца: lr=0.05, depth=3, n_estimators=127.


## Ячейка 8. Итоговое сравнение трёх моделей (уровень B, как на слайде 8.7)

Фиксируем параметры XGBoost из примера на слайде, сравниваем на том же `test`. Для Logistic Regression числовые колонки стандартизируются; деревьям это не требуется.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaled = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
    remainder=StandardScaler()
)

models = {
    "Логрегрессия": make_pipeline(scaled, LogisticRegression(max_iter=1000)),
    "Случайный лес": make_pipeline(make_prep(), RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=2)),
    "XGBoost": make_pipeline(make_prep(), XGBClassifier(
        n_estimators=127, learning_rate=0.05, max_depth=3,
        random_state=42, n_jobs=2, eval_metric="logloss")),
}
results_8_7.clear()
for name, pipe in models.items():
    fit_and_report(name, pipe, save=True)

comparison_3 = pd.DataFrame(results_8_7)
print("\nТаблица сравнения трёх моделей:")
display(comparison_3.style.format({
    "F1 train": "{:.3f}", "F1 test": "{:.3f}",
    "ROC-AUC test": "{:.3f}", "Время, с": "{:.2f}"
}))

Логрегрессия     F1 train:0.594  F1 test:0.611  AUC test:0.806  0.01 с


Случайный лес    F1 train:1.000  F1 test:0.556  AUC test:0.752  0.26 с
XGBoost          F1 train:0.644  F1 test:0.577  AUC test:0.792  0.03 с

Таблица сравнения трёх моделей:


,Модель,F1 train,F1 test,ROC-AUC test,"Время, с"
0,Логрегрессия,0.594,0.611,0.806,0.01
1,Случайный лес,1.000,0.556,0.752,0.26
2,XGBoost,0.644,0.577,0.792,0.03


## Уровень C — открытое задание: XGBoost, LightGBM и CatBoost

**Задача:** сравнить библиотеки для предсказания оттока на том же train/test.

- В XGBoost и LightGBM признаки `region` и `tariff` кодируются с помощью **One-Hot Encoding**.
- В CatBoost эти признаки используются **как исходные категории** (`cat_features`), без one-hot.
- Для сравнимости выбраны близкие настройки: **127 деревьев**, `learning_rate=0.05`, глубина **3**, `2` вычислительных потока.
- Для времени обучения и предсказания **5 запусков**, первый отбрасываем, по оставшимся четырём берём **медиану**.
- В AUC используется одна и та же тестовая выборка. Время зависит от компьютера, версии библиотек и фоновой нагрузки; значения не обязаны совпасть со слайдом.
- **Размер модели** — размер сериализованной модели (включая препроцессор, если он есть) в КиБ.

По итогам сравнения ниже появится таблица с колонками из последнего слайда.

In [9]:
import pickle
import warnings
from statistics import median
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Одинаковые для 3 моделей: число деревьев, learning_rate, глубина и число потоков.
# One-hot для XGBoost и LightGBM; нативные категории для CatBoost.

def create_benchmark_models():
    return {
        "XGBoost": make_pipeline(make_prep(), XGBClassifier(
            n_estimators=127, learning_rate=0.05, max_depth=3,
            random_state=42, n_jobs=2, eval_metric="logloss")),
        "LightGBM": make_pipeline(make_prep(), LGBMClassifier(
            n_estimators=127, learning_rate=0.05, max_depth=3,
            num_leaves=7, random_state=42, n_jobs=2,
            verbosity=-1)),
        "CatBoost": CatBoostClassifier(
            iterations=127, learning_rate=0.05, depth=3,
            loss_function="Logloss", random_seed=42, thread_count=2,
            cat_features=["region", "tariff"],
            verbose=False, allow_writing_files=False)
    }

benchmark_results = []
for name, model in create_benchmark_models().items():
    train_times = []
    infer_times = []
    probas = None
    for run in range(5):
        start = time.perf_counter()
        model.fit(X_train, y_train)
        t_train = time.perf_counter() - start

        start = time.perf_counter()
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="X does not have valid feature names.*")
            probas = model.predict_proba(X_test)[:, 1]
        t_infer = time.perf_counter() - start

        if run > 0:  # 1-й замер прогревочный, исключён из медианы
            train_times.append(t_train)
            infer_times.append(t_infer)

    benchmark_results.append({
        "Библиотека": name,
        "AUC test": roc_auc_score(y_test, probas),
        "Обучение, с": median(train_times),
        "Инференс, с": median(infer_times),
        "Размер модели, КиБ": len(pickle.dumps(model)) / 1024,
    })
    print(f"{name}: выполнено 5 прогонов, медианы рассчитаны (первый прогон отброшен)")

final_table = pd.DataFrame(benchmark_results)

best_auc_row = final_table.loc[final_table["AUC test"].idxmax()]
fast_train_row = final_table.loc[final_table["Обучение, с"].idxmin()]
fast_infer_row = final_table.loc[final_table["Инференс, с"].idxmin()]
print(f"\nВывод: лучшая AUC test — {best_auc_row['Библиотека']} ({best_auc_row['AUC test']:.3f}).")
print(f"Быстрее обучается: {fast_train_row['Библиотека']}; "
      f"быстрее предсказывает: {fast_infer_row['Библиотека']}.")
print("Для выбора в реальном проекте учитывайте также скорость, размер и удобство работы с категориями.")

XGBoost: выполнено 5 прогонов, медианы рассчитаны (первый прогон отброшен)
LightGBM: выполнено 5 прогонов, медианы рассчитаны (первый прогон отброшен)


CatBoost: выполнено 5 прогонов, медианы рассчитаны (первый прогон отброшен)

Вывод: лучшая AUC test — CatBoost (0.798).
Быстрее обучается: LightGBM; быстрее предсказывает: CatBoost.
Для выбора в реальном проекте учитывайте также скорость, размер и удобство работы с категориями.


## Финальная таблица для сдачи

**Одинаковый train/test, 5 замеров (первый исключён), медианы времени.** Последняя ячейка выводит таблицу по шаблону преподавателя.

In [10]:
display(
    final_table.style
    .format({
        "AUC test": "{:.3f}",
        "Обучение, с": "{:.3f}",
        "Инференс, с": "{:.4f}",
        "Размер модели, КиБ": "{:.1f}"
    })
    .set_caption("Итог: XGBoost vs LightGBM vs CatBoost — предсказание оттока")
)

,Библиотека,AUC test,"Обучение, с","Инференс, с","Размер модели, КиБ"
0,XGBoost,0.792,0.033,0.0035,146.4
1,LightGBM,0.793,0.025,0.0042,116.1
2,CatBoost,0.798,0.066,0.0009,32.1
